In [ ]:
from pynq import Overlay
import numpy as np
import re
import pprint
import time

overlay = Overlay('matmul.bit')          # loads matmul.bit + matmul.hwh
k = overlay.kernel
mmio = k.mmio
regs = overlay.ip_dict['kernel']['registers']

In [ ]:
rm = overlay.kernel.register_map

In [ ]:
# uncomment to find register names and addresses
# pprint.pprint(regs)

In [ ]:
def _region(overlay, ip, reg):
    info = overlay.ip_dict[ip]['registers'][reg]
    return info['address_offset'], info['size'] 

def mem_write(overlay, ip, reg, data, dtype):
    off, cap = _region(overlay, ip, reg)
    raw = np.ascontiguousarray(data, dtype=dtype).view(np.uint8).ravel()
    if raw.size > cap:
        raise ValueError(f"{reg}: {raw.size} bytes does not fit in {cap} bytes")
    raw = np.concatenate([raw, np.zeros((-raw.size) % 4, np.uint8)])   # pad to whole 32-bit words
    words = raw.view(np.uint32)
    mmio = getattr(overlay, ip).mmio
    mmio.array[off // 4 : off // 4 + words.size] = words

def mem_read(overlay, ip, reg, dtype, shape):
    off, cap = _region(overlay, ip, reg)
    n_bytes = int(np.prod(shape)) * np.dtype(dtype).itemsize
    if n_bytes > cap:
        raise ValueError(f"{reg}: {n_bytes} bytes requested, region is {cap} bytes")
    n_words = (n_bytes + 3) // 4
    mmio = getattr(overlay, ip).mmio
    words = mmio.array[off // 4 : off // 4 + n_words].copy()
    return words.view(np.uint8)[:n_bytes].view(dtype).reshape(shape)

In [ ]:
A = np.zeros((8, 8), dtype=np.int16)
B = np.zeros((8, 8), dtype=np.int16)

A[:3, :3] = [[5, 2, 3], 
             [4, 5, 6], 
             [7, 8, 9]]

B[:3, :3] = [[2, 0, 1], 
             [1, 3, 0], 
             [0, 1, 4]] 

mem_write(overlay, 'kernel', 'Memory_a', A, np.int16)
mem_write(overlay, 'kernel', 'Memory_b', B, np.int16)

t0 = time.perf_counter()
rm.CTRL.AP_START = 1
while not rm.CTRL.AP_DONE:
    pass
t1 = time.perf_counter()

C = mem_read(overlay, 'kernel', 'Memory_c', np.int64, (8,8))
print(C)
print(np.array_equal(C, A.astype(np.int32) @ B.astype(np.int32)))
print(f"time taken: {t1-t0}s")